# 长期记忆 (Long-Term Memory) 完整教程

## 概述

长期记忆提供跨会话的持久存储，使 AI Agent 能够：

1. **记住用户信息** - 姓名、偏好、历史交互
2. **存储领域知识** - 专业知识、FAQ、文档
3. **语义检索** - 基于含义而非关键词匹配

本教程涵盖：
- 向量嵌入原理
- 记忆存储与检索
- 记忆类型分类
- 持久化与加载

---

## 环境准备

In [ ]:
import sys
sys.path.insert(0, '../src')

from long_term_memory import (
    MemoryType, MemoryEntry,
    SimpleEmbedding, InMemoryVectorStore,
    LongTermMemory
)
import math
from datetime import datetime, timedelta

print("模块导入成功!")

---

## 1. 向量嵌入基础

### 1.1 什么是向量嵌入？

向量嵌入将文本映射到高维向量空间，语义相似的文本在空间中距离更近。

In [ ]:
# 创建嵌入器
embedder = SimpleEmbedding(dim=64)

# 嵌入文本
text1 = "Python 编程语言"
text2 = "Python 代码开发"
text3 = "烹饪美食食谱"

vec1 = embedder.embed(text1)
vec2 = embedder.embed(text2)
vec3 = embedder.embed(text3)

print(f"向量维度: {len(vec1)}")
print(f"向量示例 (前5维): {vec1[:5]}")

### 1.2 余弦相似度

In [ ]:
def cosine_similarity(a, b):
    """计算余弦相似度"""
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0

# 计算相似度
sim_12 = cosine_similarity(vec1, vec2)
sim_13 = cosine_similarity(vec1, vec3)

print("=== 相似度对比 ===")
print(f"'{text1}' vs '{text2}': {sim_12:.4f}")
print(f"'{text1}' vs '{text3}': {sim_13:.4f}")
print(f"\n结论: 编程相关文本相似度更高!")

---

## 2. 记忆条目 (MemoryEntry)

### 2.1 创建记忆条目

In [ ]:
# 创建记忆条目
entry = MemoryEntry(
    content="用户张三喜欢深色模式界面",
    memory_type=MemoryType.PREFERENCE,
    importance=0.8,
    metadata={"user_id": "user_001"},
    source="conversation"
)

print("=== 记忆条目属性 ===")
print(f"ID:       {entry.id}")
print(f"内容:     {entry.content}")
print(f"类型:     {entry.memory_type.value}")
print(f"重要性:   {entry.importance}")
print(f"时间戳:   {entry.timestamp}")
print(f"来源:     {entry.source}")

### 2.2 记忆类型

In [ ]:
# 展示所有记忆类型
print("=== 记忆类型 ===")
examples = {
    MemoryType.FACT: "用户的邮箱是 user@example.com",
    MemoryType.EVENT: "2024年1月15日用户购买了产品A",
    MemoryType.PREFERENCE: "用户喜欢简洁的回复风格",
    MemoryType.KNOWLEDGE: "Python 的 GIL 限制多线程性能",
    MemoryType.CONVERSATION: "用户表示对价格敏感",
    MemoryType.TASK: "提醒用户明天下午3点开会",
}

for mem_type, example in examples.items():
    print(f"  {mem_type.value:12} | {example}")

### 2.3 访问记录

In [ ]:
# 记录访问
print(f"访问前 - 访问次数: {entry.access_count}, 最后访问: {entry.last_accessed}")

entry.record_access()
entry.record_access()
entry.record_access()

print(f"访问后 - 访问次数: {entry.access_count}, 最后访问: {entry.last_accessed}")

---

## 3. 长期记忆系统

### 3.1 基本操作

In [ ]:
# 创建长期记忆
ltm = LongTermMemory()

# 存储记忆
memories = [
    ("用户名是张三", MemoryType.FACT, 0.9),
    ("用户喜欢 Python 编程", MemoryType.PREFERENCE, 0.8),
    ("用户在北京工作", MemoryType.FACT, 0.7),
    ("Python 是解释型语言", MemoryType.KNOWLEDGE, 0.6),
    ("用户昨天询问了机器学习", MemoryType.EVENT, 0.5),
]

for content, mem_type, importance in memories:
    ltm.store(content, memory_type=mem_type, importance=importance)

print(f"存储了 {ltm.size} 条记忆")

### 3.2 语义检索

In [ ]:
# 检索相关记忆
query = "用户的编程偏好"
results = ltm.recall(query, k=3)

print(f"查询: '{query}'")
print("\n=== 检索结果 ===")
for entry, similarity in results:
    print(f"  [{similarity:.3f}] {entry.content}")

In [ ]:
# 按类型过滤检索
results = ltm.recall("用户信息", k=5, memory_type=MemoryType.FACT)

print("=== 只检索 FACT 类型 ===")
for entry, similarity in results:
    print(f"  [{entry.memory_type.value}] {entry.content}")

### 3.3 列出所有记忆

In [ ]:
# 列出所有记忆
print("=== 所有记忆 ===")
for entry in ltm.list_all():
    print(f"  [{entry.memory_type.value:12}] (重要性:{entry.importance:.1f}) {entry.content}")

In [ ]:
# 按重要性过滤
important = ltm.list_all(min_importance=0.7)
print(f"\n重要性 >= 0.7 的记忆: {len(important)} 条")
for entry in important:
    print(f"  {entry.content}")

### 3.4 删除记忆

In [ ]:
# 获取一条记忆并删除
entries = ltm.list_all()
if entries:
    target = entries[0]
    print(f"删除前记忆数: {ltm.size}")
    print(f"删除: {target.content}")
    
    ltm.forget(target.id)
    print(f"删除后记忆数: {ltm.size}")

---

## 4. 持久化存储

### 4.1 保存到文件

In [ ]:
import tempfile
import os

# 保存记忆
save_path = os.path.join(tempfile.gettempdir(), "test_memories.json")
ltm.save(save_path)
print(f"记忆已保存到: {save_path}")

### 4.2 从文件加载

In [ ]:
# 创建新实例并加载
ltm2 = LongTermMemory()
count = ltm2.load(save_path)

print(f"加载了 {count} 条记忆")
print("\n=== 加载的记忆 ===")
for entry in ltm2.list_all():
    print(f"  {entry.content}")

---

## 5. 实际应用示例

### 5.1 用户画像系统

In [ ]:
class UserProfile:
    """用户画像管理"""
    
    def __init__(self, user_id: str):
        self.user_id = user_id
        self.memory = LongTermMemory()
    
    def learn(self, info: str, info_type: MemoryType = MemoryType.FACT):
        """学习用户信息"""
        self.memory.store(
            content=info,
            memory_type=info_type,
            metadata={"user_id": self.user_id}
        )
    
    def recall(self, query: str, k: int = 3):
        """回忆相关信息"""
        return self.memory.recall(query, k=k)
    
    def get_preferences(self):
        """获取用户偏好"""
        return self.memory.list_all(memory_type=MemoryType.PREFERENCE)

# 使用示例
profile = UserProfile("user_001")

# 学习用户信息
profile.learn("用户名是李四", MemoryType.FACT)
profile.learn("用户喜欢深色主题", MemoryType.PREFERENCE)
profile.learn("用户偏好简洁回复", MemoryType.PREFERENCE)
profile.learn("用户是软件工程师", MemoryType.FACT)

# 查询
print("=== 用户偏好 ===")
for pref in profile.get_preferences():
    print(f"  {pref.content}")

---

## 总结

本教程介绍了长期记忆系统的核心概念：

1. **向量嵌入**: 将文本映射到向量空间
2. **语义检索**: 基于含义而非关键词
3. **记忆分类**: FACT, EVENT, PREFERENCE 等
4. **持久化**: 保存和加载记忆

长期记忆使 AI Agent 能够跨会话记住用户信息和领域知识。